In [1]:
!pip install -q transformers torch gradio

In [2]:
from transformers import pipeline
import gradio as gr

In [5]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

print("Question Answering model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question Answering model loaded successfully!


In [6]:
def answer_question(context, question):

    if not context.strip():
        return "Please enter a context."

    if not question.strip():
        return "Please enter a question."

    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start_scores = outputs.start_logits
    end_scores = outputs.end_logits

    start_index = torch.argmax(start_scores)
    end_index = torch.argmax(end_scores)

    if end_index < start_index:
        end_index = start_index

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    confidence = (
        torch.softmax(start_scores, dim=-1)[0][start_index] *
        torch.softmax(end_scores, dim=-1)[0][end_index]
    ).item()

    return f"Answer: {answer}\nConfidence: {confidence:.2%}"

In [7]:
def answer_question(context, question):

    if not context.strip():
        return "Please enter a context."

    if not question.strip():
        return "Please enter a question."

    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start_scores = outputs.start_logits
    end_scores = outputs.end_logits

    start_index = torch.argmax(start_scores)
    end_index = torch.argmax(end_scores)

    if end_index < start_index:
        end_index = start_index

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    confidence = (
        torch.softmax(start_scores, dim=-1)[0][start_index] *
        torch.softmax(end_scores, dim=-1)[0][end_index]
    ).item()

    return f"Answer: {answer}\nConfidence: {confidence:.2%}"

In [8]:
context = """
Machine Learning is a subset of Artificial Intelligence.
It allows computers to learn patterns from data and make predictions
without being explicitly programmed for every task.
"""

question = "What is Machine Learning?"

print(answer_question(context, question))

Answer: a subset of Artificial Intelligence
Confidence: 45.83%


In [9]:
with gr.Blocks() as app:

    gr.Markdown("# 🤖 Question Answering System")

    gr.Markdown(
        "Enter a passage and ask a question. "
        "The AI will find the answer from the passage."
    )

    context_input = gr.Textbox(
        label="Context / Passage",
        lines=10
    )

    question_input = gr.Textbox(
        label="Question"
    )

    answer_button = gr.Button("Get Answer")

    output = gr.Textbox(
        label="Answer"
    )

    answer_button.click(
        fn=answer_question,
        inputs=[context_input, question_input],
        outputs=output
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c487d0ed23c708404.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
